In [3]:
import os
import pandas as pd
import tkinter as tk
from tkinter import filedialog
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder

class ROS2BagMCAPToCSV:
    def __init__(self):
        self.decoder = Decoder()


    @staticmethod
    def flatten_dict(d, parent_key='', sep='.'):
        items = []

        # 1. Intentar convertir a diccionario de todas las formas posibles
        if hasattr(d, '__getstate__'):
            data_map = d.__getstate__()
        elif hasattr(d, 'get_fields_and_field_types'): # Específico de algunas versiones de ROS2
            data_map = {field: getattr(d, field) for field in d.get_fields_and_field_types().keys()}
        elif hasattr(d, '__dict__'):
            data_map = vars(d)
        elif isinstance(d, dict):
            data_map = d
        else:
            # Si llegamos aquí y es un objeto, intentamos ver si tiene un .data (común en String.msg)
            if hasattr(d, 'data'):
                return {parent_key: str(d.data)}
            return {parent_key: str(d)}

        for k, v in data_map.items():
            if k.startswith('_'): continue

            new_key = f"{parent_key}{sep}{k}" if parent_key else k

            # 2. Análisis del valor 'v'
            # Si es un objeto complejo (pero no un string/número/bytes)
            if hasattr(v, '__dict__') or hasattr(v, '__getstate__') or isinstance(v, dict):
                # Caso especial: Si el objeto tiene un .data que es el valor real
                if hasattr(v, 'data') and not (hasattr(v.data, '__dict__') or isinstance(v.data, dict)):
                    val = v.data
                    if isinstance(val, bytes): val = val.decode('utf-8', errors='ignore')
                    items.append((new_key, val))
                else:
                    items.extend(ROS2BagMCAPToCSV.flatten_dict(v, new_key, sep=sep).items())

            # 3. Tratamiento de tipos básicos
            elif isinstance(v, bytes):
                items.append((new_key, v.decode('utf-8', errors='ignore')))
            elif isinstance(v, list) or isinstance(v, tuple):
                # Si la lista contiene bytes, decodificarlos
                clean_list = [x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else x for x in v]
                items.append((new_key, str(clean_list)))
            elif v is None:
                items.append((new_key, ""))
            else:
                # Forzamos que sea un tipo básico de Python
                items.append((new_key, v))

        return dict(items)




    def process_single_file(self, mcap_path):
        output_dir = os.path.splitext(mcap_path)[0] + "_csv"
        os.makedirs(output_dir, exist_ok=True)
        
        data_by_topic = {}
        
        try:
            with open(mcap_path, "rb") as f:
                reader = make_reader(f)
                for schema, channel, message in reader.iter_messages():
                    topic_name = channel.topic
                    ros_msg = self.decoder.decode(schema, message)

                    # Filtrar manualmente solo los campos de datos útiles
                    msg_dict = {
                        slot: getattr(ros_msg, slot) 
                        for slot in dir(ros_msg) 
                        if not slot.startswith('_') and not callable(getattr(ros_msg, slot))
                    }

                    record = {
                        "log_time_s": message.log_time / 1e9,
                        "publish_time_s": message.publish_time / 1e9,
                        **self.flatten_dict(msg_dict)
                    }

                    if topic_name not in data_by_topic:
                        data_by_topic[topic_name] = []
                    data_by_topic[topic_name].append(record)

            for topic_name, records in data_by_topic.items():
                if not records: continue
                
                df = pd.DataFrame(records)
                clean_name = topic_name.strip("/").replace("/", "_")
                
                # Exportación robusta
                df.to_csv(
                    os.path.join(output_dir, f"{clean_name}.csv"), 
                    index=False,
                    quoting=1,       # Pone comillas a todos los strings (evita errores de lectura)
                    decimal='.',     # Fuerza el punto decimal para floats
                    escapechar='\\',
                    #quotechar ='"',
                    encoding='utf-8-sig' # Asegura que los textos no se corrompan
                )
            
            print(f"✅ Procesado con éxito: {os.path.basename(mcap_path)}")
        except Exception as e:
            print(f"❌ Error procesando {mcap_path}: {e}")
            
    def run_recursive(self, root_folder):
        """Busca y procesa todos los .mcap en la carpeta y subcarpetas."""
        mcap_files = []
        for root, dirs, files in os.walk(root_folder):
            for file in files:
                if file.endswith(".mcap"):
                    mcap_files.append(os.path.join(root, file))
        
        if not mcap_files:
            print("No se encontraron archivos .mcap en el directorio seleccionado.")
            return

        print(f"🔍 Se encontraron {len(mcap_files)} archivos. Iniciando conversión...")
        for path in mcap_files:
            print(f"🚀 Procesando: {path}")
            self.process_single_file(path)
        print("\n✨ ¡Proceso finalizado!")

def main():
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    folder_path = filedialog.askdirectory(title="Selecciona la carpeta raíz para buscar MCAPs")

    if folder_path:
        converter = ROS2BagMCAPToCSV()
        converter.run_recursive(folder_path)
    else:
        print("Operación cancelada.")

if __name__ == '__main__':
    main()

🔍 Se encontraron 6 archivos. Iniciando conversión...
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251124_094548\20251124_094548_0.mcap
✅ Procesado con éxito: 20251124_094548_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_111941\20251217_111941_0.mcap
✅ Procesado con éxito: 20251217_111941_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_120944\20251217_120944_0.mcap
✅ Procesado con éxito: 20251217_120944_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_125205\20251217_125205_0.mcap
✅ Procesado con éxito: 20251217_125205_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_125816\20251217_125816_0.mcap
✅ Procesado con éxito: 20251217_125816_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_131